# Trabajo Práctico 2 - Grupo 02

### Modelo Random Forest — Entrega N6 (definitiva)

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

## Objetivo

Construir el **Random Forest definitivo** del grupo, superando de forma sustancial el techo de las entregas anteriores (mejor RF previo: **F1-macro 0.6529** en validación, v5).

El diagnóstico de las versiones N1-N5 es claro: el cuello de botella **no fue el modelo sino el desbalance de clases**. Todas entrenaron sobre el `train.csv` original (40/20/40) con `class_weight="balanced"`, y la clase **neutra** quedó siempre clavada en F1 ~0.42. El mismo problema lo resolvió XGBoost: entrenando sobre un dataset balanceado por *data augmentation* saltó de 0.66 a **0.75**, con la neutra subiendo a ~0.68.

Esta entrega aplica a Random Forest las tres palancas que **nunca se habían combinado** sobre este modelo:

1. **Dataset balanceado** (`train_augmented_llm_balanced.csv`, ~67k filas, ~33/33/33): reseñas sintéticas generadas por LLM (Llama-3.1-8b) que rellenan la clase neutra.
2. **Meta-features densas** de `extract_features` (longitud, signos `!`/`?`, mayúsculas, negaciones, SentiWordNet) concatenadas al TF-IDF mediante `VectorizerPlusFeaturesPipeline`. Aportan señales numéricas que los árboles aprovechan en los splits.
3. **Búsqueda de hiperparámetros amplia y local**: aprovechando los 12 threads del Ryzen 5 9600X y sin los límites de RAM/timeout de Colab que truncaban las búsquedas anteriores (la v4.2 murió con `KeyboardInterrupt`). Se exploran bosques grandes (hasta 800 árboles).

**Nota sobre la GPU (9070 XT):** Random Forest de scikit-learn corre íntegramente en CPU; no existe implementación de RF en GPU para AMD/ROCm (cuML es exclusivo de NVIDIA). El máximo aprovechamiento de la PC para este modelo es el paralelismo multinúcleo del procesador, no la placa de video.

**Estrategia de cómputo:** `extract_features` usa SentiWordNet token a token y es costoso. Calcularlo dentro del `RandomizedSearchCV` (con CV) lo recomputaría cientos de veces. Por eso se **precomputa una sola vez** la matriz `TF-IDF + meta-features` y la búsqueda tunea únicamente el Random Forest. El artefacto final sí se envuelve en el `VectorizerPlusFeaturesPipeline` reutilizable (la extracción lenta corre solo en el reentrenamiento final).

## Importación e instalación de dependencias

In [ ]:
# Entrenamiento local. Si faltan dependencias, descomentar:
# !pip install -q spacy scikit-learn nltk scipy pandas numpy
# !python -m spacy download es_core_news_sm

In [ ]:
import time
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
import sys
# v6 -> Random Forest -> Entregas -> TP2 (donde vive el paquete common/)
sys.path.insert(0, "../../../")

In [ ]:
from common.preprocessing import clean_classical
from common.data_utils import (
    get_split, make_tfidf, extract_features,
    VectorizerPlusFeaturesPipeline, SEED,
)
from common.evaluation import evaluate
from common.io_utils import save_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

Se usa el dataset balanceado por LLM. El split estratificado (`get_split`) preserva la proporción ~33/33/33 en train y validación, por lo que el F1-macro de validación es comparable al de XGBoost sobre el mismo dataset.

Se conservan **dos versiones del texto**:
- `*_clean`: limpiado con `clean_classical` (lematización + stopwords) → entra al TF-IDF.
- `*_raw`: texto crudo original → entra a `extract_features` (longitud, puntuación, etc. necesitan el texto sin tocar).

In [ ]:
train = pd.read_csv("../../../data/train_augmented_llm_balanced.csv")
test  = pd.read_csv("../../../data/test.csv")

print(f"Train aumentado (LLM): {len(train):,} filas")
print(train['label'].value_counts(normalize=True).sort_index().round(3))

# X_*_raw quedan como texto CRUDO (para extract_features)
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [ ]:
print("Preprocesando (clean_classical)... puede tardar unos minutos sobre 67k filas")
t0 = time.time()
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
print(f"Tiempo: {time.time()-t0:.0f}s")

## Precómputo de la matriz TF-IDF + meta-features

El TF-IDF se fija a la mejor configuración hallada en las entregas previas (unigramas+bigramas, `sublinear_tf=True`), validada repetidamente en N2-N5. Sobre esa matriz dispersa se concatenan, con `scipy.sparse.hstack`, las 12 meta-features densas de `extract_features`. Random Forest es invariante a la escala (parte por umbrales), así que no hace falta estandarizar las columnas numéricas.

Se calcula **una sola vez** para train, validación y test; la búsqueda de hiperparámetros reutiliza estas matrices.

In [ ]:
tfidf = make_tfidf()  # ngram (1,2), min_df=5, max_df=0.95, sublinear_tf=True, max_features=100k

Xtr_txt = tfidf.fit_transform(X_train)
Xva_txt = tfidf.transform(X_val)
print(f"TF-IDF: {Xtr_txt.shape[1]:,} features de texto")

print("Extrayendo meta-features (SentiWordNet es lento, paciencia)...")
t0 = time.time()
feat_tr = csr_matrix(extract_features(X_train_raw))
feat_va = csr_matrix(extract_features(X_val_raw))
print(f"Meta-features: {feat_tr.shape[1]} columnas | tiempo: {time.time()-t0:.0f}s")

Xtr = hstack([Xtr_txt, feat_tr]).tocsr()
Xva = hstack([Xva_txt, feat_va]).tocsr()
print(f"Matriz combinada train: {Xtr.shape}")

## Búsqueda de hiperparámetros (RandomizedSearchCV)

Búsqueda amplia sobre Random Forest, con `class_weight="balanced"` (refuerza el balanceo ya logrado por el dataset). Se exploran:

| Parámetro | Valores |
|-----------|---------|
| n_estimators | 400, 600, 800 |
| max_depth | None, 80 |
| min_samples_leaf | 1, 2, 4 |
| min_samples_split | 2, 5, 10 |
| max_features | sqrt, log2 |

`n_jobs=-1` en el RF usa los 12 threads dentro de cada ajuste; el search corre los candidatos secuencialmente (`n_jobs=1`) para no multiplicar el uso de RAM con matrices de 100k columnas. Subí `n_iter` si tu RAM lo permite.

In [ ]:
rf = RandomForestClassifier(
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)

param_dist = {
    "n_estimators":      [400, 600, 800],
    "max_depth":         [None, 80],
    "min_samples_leaf":  [1, 2, 4],
    "min_samples_split": [2, 5, 10],
    "max_features":      ["sqrt", "log2"],
}

rs = RandomizedSearchCV(
    rf, param_dist,
    n_iter=24, cv=4,
    scoring="f1_macro",
    n_jobs=1,
    random_state=SEED,
    verbose=2,
)

print("Buscando mejores hiperparámetros (esto tarda; usa todos los núcleos)...")
t0 = time.time()
rs.fit(Xtr, y_train)
print(f"\nTiempo total de búsqueda: {(time.time()-t0)/60:.1f} min")
print(f"Mejores hiperparámetros: {rs.best_params_}")
print(f"Mejor F1-macro en CV: {rs.best_score_:.4f}")

In [ ]:
y_pred = rs.best_estimator_.predict(Xva)
evaluate("rf_tfidf_feats_llm_v6", y_val, y_pred, hyperparams=rs.best_params_)

## Modelo final: artefacto reutilizable, reentrenamiento y submission

Se construye un `VectorizerPlusFeaturesPipeline` autocontenido (TF-IDF + meta-features + RF con los mejores hiperparámetros) y se reentrena sobre **todo** el dataset balanceado (train + val). Este objeto único hace `fit`/`predict` desde el texto, por lo que el equipo docente puede cargarlo con `load_model` y predecir sin pasos previos.

La extracción lenta de features solo corre acá (una vez), no durante la búsqueda.

In [ ]:
# Texto crudo y limpio completos (train + val)
X_full_clean = np.concatenate([X_train, X_val])
X_full_raw   = np.concatenate([X_train_raw, X_val_raw])
y_full       = np.concatenate([y_train, y_val])

final_rf = RandomForestClassifier(
    **rs.best_params_,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)

# VectorizerPlusFeaturesPipeline espera X = column_stack([texto_limpio, texto_crudo])
final_model = VectorizerPlusFeaturesPipeline(vectorizer=make_tfidf(), classifier=final_rf)

print("Reentrenando modelo final sobre train + val completos...")
t0 = time.time()
final_model.fit(np.column_stack([X_full_clean, X_full_raw]), y_full)
print(f"Tiempo: {(time.time()-t0)/60:.1f} min")

In [ ]:
save_model(final_model, "rf_tfidf_feats_llm_v6")

# make_submission pasa raw_texts como segunda columna para extract_features
make_submission(
    final_model,
    pd.DataFrame({"id": test["id"], "text": X_test}),
    "submission_rf_tfidf_feats_llm_v6.csv",
    raw_texts=test["text"].values,
);